Temporal Semantic Novelty

Computes:
- Strict novelty  = 1 - max cosine similarity to prior work
- kNN novelty     = 1 - mean(top-k similarities to prior work)

Both computed in a single temporal pass.

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

In [3]:
## Load embeddings and metadata
paper_ids = np.load('../../outputs/intermediate/paper_ids.npy')
embeddings = np.load('../../outputs/intermediate/abstract_embeddings.npy')

meta = pd.read_csv('../../outputs/intermediate/openalex_metadata_full.csv')

meta = meta[meta['global_paper_id'].isin(paper_ids)]
meta = meta.set_index('global_paper_id').loc[paper_ids].reset_index()

years = meta['year'].values

In [4]:
## Sort by year (temporal order)
sorted_idx = np.argsort(years)

embeddings = embeddings[sorted_idx]
paper_ids = paper_ids[sorted_idx]
years = years[sorted_idx]

In [45]:
## Compute Novelty
top_k = 5
n = len(embeddings)

semantic_strict = np.zeros(n)
semantic_knn = np.zeros(n)

for i in tqdm(range(n)):
    cur_emb = embeddings[i].reshape(1, -1)
    cur_years = years[i]

    prev_mask = years < cur_years

    if not prev_mask.any():
        semantic_strict[i] = 1.0
        semantic_knn[i] = 1.0
        continue

    prev_embeddings = embeddings[prev_mask]
    sims = cosine_similarity(cur_emb, prev_embeddings)[0]       # Calc. The sim.

    #Strict Novelty
    max_sim = np.max(sims)          # most similar prior paper
    semantic_strict[i] = 1-max_sim

    #kNN novelty
    k = min(top_k, len(sims))
    top_sim = np.partition(sims, -k)[-k:]
    semantic_knn[i] = np.mean(top_sim)

100%|██████████| 2511/2511 [00:15<00:00, 167.12it/s]


In [46]:
## Clip float values
semantic_strict = np.clip(semantic_strict, 0, 1)
semantic_knn = np.clip(semantic_knn, 0, 1)

In [47]:
print(semantic_strict)
print(semantic_knn)

[1.         1.         1.         ... 0.05570769 0.04864728 0.38414133]
[1.         1.         1.         ... 0.94163477 0.94680578 0.61585855]


In [48]:
## Save output
strict_df = pd.DataFrame({
    'paper_id': paper_ids,
    'year': years,
    'semantic_strict': semantic_strict
})

kNN_df = pd.DataFrame({
    'paper_id': paper_ids,
    'year': years,
    'semantic_knn': semantic_knn
})

strict_df.to_csv("../../outputs/semantic_novelty_scores.csv", index=False)
kNN_df.to_csv("../../outputs/semantic_novelty_knn_scores.csv", index=False)

In [49]:
pd.read_csv('../../outputs/final/semantic_novelty_scores.csv').describe()

,year,semantic_strict
count,2511.000000,2511.000000
mean,2017.811629,0.077135
std,2.955035,0.144783
min,2010.000000,0.000000
25%,2016.000000,0.048051
50%,2019.000000,0.055526
75%,2020.000000,0.063785
max,2025.000000,1.000000


In [50]:
pd.read_csv('../../outputs/final/semantic_novelty_scores.csv')['semantic_strict'].value_counts()

semantic_strict
1.000000    59
0.000000     9
0.069771     3
0.058846     2
0.070377     2
            ..
0.067936     1
0.071369     1
0.034524     1
0.071118     1
0.023855     1
Name: count, Length: 2379, dtype: int64

In [51]:
pd.read_csv('../../outputs/final/semantic_novelty_knn_scores.csv')['semantic_knn'].value_counts()

semantic_knn
1.000000    63
0.775232     3
0.950478     3
0.927262     3
0.943043     2
            ..
0.925050     1
0.919877     1
0.923607     1
0.935964     1
0.903002     1
Name: count, Length: 2362, dtype: int64